In [24]:
import pandas as pd
import numpy as np
import altair as alt
import matplotlib.pyplot as plt

In [25]:
data_url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/building_inventory.csv"
df = pd.read_csv(data_url)
df

,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
0,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,1975,1975,144,1,1,0,Unusual,Unusual,Not provided
1,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
2,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
3,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
4,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8857,Department of Transportation,Belvidere Maintenance Storage Facility - Boone...,9797 Illinois Rte. 76,Belvidere,61008,Boone,16,Adam Kinzinger,69,Sosnowski Joe,...,In Use,0,0,432,1,0,0,Storage,NaN,NaN
8858,Department of Transportation,Belvidere Maintenance Storage Facility - Boone...,9797 Illinois Rte 76,Belvidere,61008,Boone,16,Adam Kinzinger,69,Sosnowski Joe,...,In Use,0,0,330,1,0,0,Storage,NaN,NaN
8859,Department of Transportation,Quincy Maintenance Storage Facility,800 Koch's Lane,Quincy,62305,Adams,18,Darin M. LaHood,94,Frese Randy E.,...,In Use,0,1987,130,1,0,0,Storage,High Hazard,NaN
8860,Illinois Community College Board,Illinois Valley Community College - Oglesby,815 North Orlando Smith Avenue,Oglesby,61348,LaSalle,16,Adam Kinzinger,76,Long Jerry Lee,...,In Use,1971,1971,49552,1,1,0,Education,Education,Not provided


In [26]:
df.loc[df['Year Acquired'] == 0, 'Year Acquired'] = np.nan
df.loc[df['Square Footage'] == 0, 'Square Footage'] = np.nan

In [27]:
stats = df.groupby('Year Acquired')['Square Footage'].describe()
stats

,count,mean,std,min,25%,50%,75%,max
Year Acquired,,,,,,,,
1753.0,1.0,1200.000000,NaN,1200.0,1200.0,1200.0,1200.00,1200.0
1802.0,2.0,2220.000000,1943.129435,846.0,1533.0,2220.0,2907.00,3594.0
1810.0,3.0,1344.333333,1809.945948,216.0,300.5,385.0,1908.50,3432.0
1832.0,1.0,120000.000000,NaN,120000.0,120000.0,120000.0,120000.00,120000.0
1837.0,1.0,10302.000000,NaN,10302.0,10302.0,10302.0,10302.00,10302.0
...,...,...,...,...,...,...,...,...
2015.0,20.0,15254.650000,29153.085290,144.0,696.0,3152.0,10590.25,105000.0
2016.0,10.0,30483.900000,61864.180491,1152.0,2464.0,3352.5,3793.00,184000.0
2017.0,1.0,6720.000000,NaN,6720.0,6720.0,6720.0,6720.00,6720.0


In [28]:
stats.index = pd.to_datetime(stats.index.astype('int'), format='%Y')
stats_melt = stats.reset_index().melt('Year Acquired',var_name='Statistic',value_name='Stats Value')
stats_melt


,Year Acquired,Statistic,Stats Value
0,1753-01-01,count,1.0
1,1802-01-01,count,2.0
2,1810-01-01,count,3.0
3,1832-01-01,count,1.0
4,1837-01-01,count,1.0
...,...,...,...
1363,2015-01-01,max,105000.0
1364,2016-01-01,max,184000.0
1365,2017-01-01,max,6720.0
1366,2018-01-01,max,12000.0


In [29]:
stats_melt.loc[stats_melt['Stats Value'] == 0, 'Stats Value'] = np.nan

In [30]:
plot1 = alt.Chart(stats_melt).mark_line().encode(
    alt.X('Year Acquired:T'),
    alt.Y('Stats Value:Q', scale=alt.Scale(type='log')),
    color='Statistic:N'
).properties(
    height=300,
    width=700,
    title='Building Square Footage Statistics Over Time'
)

plot1

alt.Chart(...)

In [31]:
dropdown = alt.binding_select(options=['50%','mean','min','max'], name='Statistic Selection: ')
selection = alt.selection_point(fields=['Statistic'], bind=dropdown)

color = alt.condition(selection,
                      alt.Color('Statistic:N'),
                      alt.value('lightgray'))

opacity = alt.condition(selection, alt.value(1.0), alt.value(0.25))

In [32]:
plot2 = alt.Chart(stats_melt).mark_line().encode(
    alt.X('Year Acquired:T'),
    alt.Y('Stats Value:Q', scale=alt.Scale(type='log')),
    color=color,
    opacity=opacity
).properties(
    height=300,
    width=700,
    title='Interactive Building Statistics'
).add_params(
    selection
)

plot2

alt.Chart(...)

In [33]:
plot1.save('plot1.json')
plot2.save('plot2.json')

Plot Design: The visualization uses a line chart to show the difference of building square footage over time. A line chart is a good pick because the x-axis can represent time (Year Acquired), and that allows for the trends to be easily observed. The x-axis shows the acquisition year as a temporal variable, while the y-axis encodes the square footage statistic values. A log scale is used on the y-axis because square footage values vary. Using a log scale allows for small and large values to be visible and prevents large buildings from taking over the visualization.

Data Transformations: The dataset is grouped by Year Acquired and summary statistics of square footage. This gave values such as the min, median, mean, and max building sizes for every year. The  dataframe was then reshaped using the melt function as shown in class. This transformation created two columns, two different statistic types. This structure was ideal for Altair to draw multiple lines using a singular encoding.

Interactivity: The dropdown selection was added to allow users to choose which statistic to highlight. The dropdown is covering the statistic field and enables dynamic interaction in our plots. When the statistic is selected, the corresponding line is highlighted using color. All the other lines are faded so you can easily visualize the target. This interaction makes it easier to focus on a specific statistics while still having the rest of the data present.